In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../datasets/data.csv')
df_id_authors = df[['ID', 'Authors']].copy()

df_id_authors

,ID,Authors
0,1,"Janet M. Weisenberger, Arnold F. Heidbreder, J..."
1,2,"Stephen Brewster, Joanna Lumsden, Marek Bell, ..."
2,3,"Georgios Marentakis, Stephen A. Brewster"
3,4,"Christian Metzger, Matt Anderson, Thad Starner"
4,5,"Vincent Buil, Gerard Hollemans"
...,...,...
137,138,"Xuefu Dong, Lixing He, Zengyi Han, Kenneth Chr..."
138,139,"Daibo Liu, Hang Lu, Xiaomeng Qi, Huigui Rong, ..."
139,140,"Ken Takaki, Mitsuhiro Kamezaki, Takuya Sasatan..."
140,141,"Max J. F. van Oort, Gabriel E. Sáenz, Selina T..."


In [3]:
# Build exact co-author matrix from comma-separated author strings
coauthor_matrix = pd.DataFrame(0, index=np.arange(1, len(df_id_authors)+1), columns=np.arange(1, len(df_id_authors)+1))
ids = df_id_authors['ID'].to_numpy(dtype=int)

def normalize_name(name: str) -> str:
    # lowercase + collapse internal whitespace
    return ' '.join(name.strip().lower().split())

def to_author_set(value) -> set:
    # Accept list or single comma-separated string
    if isinstance(value, list):
        names = value
    elif isinstance(value, str):
        # split on commas that separate authors
        names = [n for n in (x.strip() for x in value.split(',')) if n]
    else:
        names = []
    return {normalize_name(n) for n in names}

# Map ID -> normalized author set (robust to missing rows)
id_to_authors = {int(row['ID']): to_author_set(row['Authors']) for _, row in df_id_authors.iterrows()}

# Only connect papers sharing at least one EXACT author name (distance == 0)
for id_i in ids:
    authors_i = id_to_authors.get(id_i, set())
    for id_j in range(id_i + 1, len(ids) + 1):
        authors_j = id_to_authors.get(id_j, set())
        if authors_i and authors_j and authors_i.intersection(authors_j):
            coauthor_matrix.loc[id_i, id_j] = 1
            coauthor_matrix.loc[id_j, id_i] = 1  # symmetric

coauthor_matrix.to_csv('../datasets/interconnections/coauthor_matrix.csv')
coauthor_matrix.head()

,1,2,3,4,5,6,7,8,9,10,...,133,134,135,136,137,138,139,140,141,142
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
